# 🚀 End-to-End Extraction, Segmentation, and AI Validation Pipeline
This notebook executes the entire zero-shot pipeline (Grounding DINO + SAM 2) and introduces a novel **VLM-as-a-Judge (Qwen2.5-VL)** step. 

Instead of manual QA, we pass the resulting segmentation masks back into the Vision Language Model and ask it to critique the segmentation quality, identifying background leakage, missing feathers, or clipped boundaries.

In [ ]:
import os
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from ultralytics import SAM
from mlx_vlm import load, generate

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Hardware Accelerator: {device}")

DATA_DIR = "../data/raw"
OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Test image
image_path = os.path.join(DATA_DIR, 'A1383 2000-im1316.jpg')

In [ ]:
def run_pipeline(img_path):
    print("1. Running Grounding DINO...")
    dino_id = 'IDEA-Research/grounding-dino-base'
    dino_processor = AutoProcessor.from_pretrained(dino_id)
    dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(dino_id).to(device)
    
    img = Image.open(img_path).convert('RGB')
    inputs = dino_processor(images=img, text='bird feather.', return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = dino_model(**inputs)
    
    results = dino_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids, target_sizes=[img.size[::-1]]
    )[0]
    
    boxes = [box.tolist() for score, box in zip(results['scores'], results['boxes']) if score > 0.45]
    del dino_model
    torch.mps.empty_cache()
    
    print(f"   Found {len(boxes)} feathers. Passing to SAM 2...")
    sam_model = SAM('sam2.1_b.pt')
    sam_results = sam_model(img_path, bboxes=boxes, device='mps', verbose=False)[0]
    
    return boxes, sam_results

boxes, sam_results = run_pipeline(image_path)

In [ ]:
def create_validation_composite(img_path, boxes, sam_res):
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    overlay = img_rgb.copy()
    
    # Draw masks in red
    if sam_res.masks is not None:
        for m in sam_res.masks.data.cpu().numpy():
            m = cv2.resize(m.astype(np.float32), (img.shape[1], img.shape[0]))
            overlay[m > 0.5] = [255, 0, 0]
            
    # Draw bounding boxes in bright green
    for box in boxes:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 6)
        
    blended = cv2.addWeighted(img_rgb, 0.4, overlay, 0.6, 0)
    
    comp_path = os.path.join(OUTPUT_DIR, "validation_composite.jpg")
    cv2.imwrite(comp_path, cv2.cvtColor(blended, cv2.COLOR_RGB2BGR))
    
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.imshow(blended)
    ax.set_title("AI Quality Assurance Composite (Masks & Bounds)")
    ax.axis('off')
    plt.show()
    
    return comp_path

composite_path = create_validation_composite(image_path, boxes, sam_results)

In [ ]:
def vlm_as_a_judge(comp_img_path):
    print("Loading Qwen2.5-VL to judge segmentation quality...")
    vlm_path = "mlx-community/Qwen2.5-VL-7B-Instruct-4bit"
    model, processor = load(vlm_path)
    
    prompt = '''
    You are an expert computer vision Quality Assurance bot. 
    Analyze the provided image. The bright green boxes represent bounding box detections. The red shaded areas represent pixel-perfect segmentation masks generated by SAM 2.
    
    Critique the quality of the segmentation. 
    1. Are the red masks covering the bird feathers entirely?
    2. Is there background leakage (red mask bleeding onto the white paper, ruler, or tape)?
    3. Provide an overall confidence score out of 10.
    
    Return EXACTLY in this JSON format:
    {
      "all_feathers_covered": true/false,
      "background_leakage_detected": true/false,
      "quality_score_1_to_10": 9,
      "critique": "Your brief explanation here."
    }
    '''
    
    print("Asking VLM to critique the composite...")
    try:
        output = generate(model, processor, prompt=prompt, image=[comp_img_path], max_tokens=256)
        print("\n=== VLM JUDGE REPORT ===")
        print(output)
    except Exception as e:
        print(f"VLM evaluation failed: {e}")

vlm_as_a_judge(composite_path)